In [9]:
import pandas as pd
import re 
import numpy as np
import nltk
# from preprocessing_functions import *
import json
import ast
from typing import List, Tuple, Optional
from geopy.geocoders import Nominatim
from geopy.exc import GeocoderTimedOut, GeocoderServiceError
import time
import math
import requests

In [3]:
data = pd.read_csv("results---From---2023-10-31--22-08-07---To---2024-04-07--16-02-28.csv")
data = data[['message']].dropna()
data = data.drop_duplicates()
data

,message
0,#Renta x DÍAS de Apto en el Vedado
1,No tienes permisos para ejecutar este comando ...
2,/revisarbrplus@ReputacionPlusBot
8,Casa en venta en la zona sur cerca de las fábr...
9,Busco renta por tiempo indefinido para una par...
...,...
4988,Busco alquiler en el vedado límite 150 verde s...
4992,"Busco alquiler por tiempo indefinido, 58316712"
4993,Busco alquiler en la lisa o lo más cerca posible
4994,Busco alquiler en la Lisa


#### Ahora vamos a empezar a extraer features de interés y vamos empezar con el precio y la moneda en que se haría la negociación

In [5]:
def extract_price(message):
    prices = re.findall(r'\b\d{2,6}(?:[.,]\d+)? | \d{2,6}(?:[.,]\d+)? mil\b', message)
    if prices:
        return prices
    return "no especificado"
def extract_currency(message):
    currencies = re.findall(r'\b(USD|usd|dólar|dolar|EURO|euro|MLC|mlc|CUP|cup|pesos|mn|dolar|dolares|mil)\b', message, re.IGNORECASE)
    if currencies:
        return currencies
    return "no especificado"

##### Vamos a ir probando

In [16]:
data['message'] = data['message'].apply(delete_emojis)
data['message'] = data['message'].apply(delete_commands) 
data['message'] = data['message'].apply(normalize_text)
data['precio'] = data['message'].apply(extract_price)
data['moneda'] = data['message'].apply(extract_currency) 

print(data)

                                                message           precio  \
0                     renta x días de apto en el vedado  no especificado   
1     no tienes permisos para ejecutar este comando ...  no especificado   
2                                                        no especificado   
8     casa en venta en la zona sur cerca de las fábr...          [2800 ]   
9     busco renta por tiempo indefinido para una par...        [ 20 mil]   
...                                                 ...              ...   
4988  busco alquiler en el vedado límite 150 verde s...           [150 ]   
4992     busco alquiler por tiempo indefinido, 58316712  no especificado   
4993   busco alquiler en la lisa o lo más cerca posible  no especificado   
4994                          busco alquiler en la lisa  no especificado   
4996  busco alquiler en playa, marianao, lisa hasta ...  no especificado   

               moneda                                        ubicaciones  \
0     no es

##### Intentemos extraer las ubicaciones usando regex, ya que el modelo preentrenado de spacy no resultó muy útil.

In [7]:
with open("barrios_calles_habana.json", "r", encoding="utf-8") as file:
    json_data = json.load(file)

streets = [r"(calle\s+\w+(?:\s+\w+)*|calzada\s+\w+(?:\s+\w+)*)"]
intersections = r"((entre|esquina|y)\s+(calle|calzada|av\.\?|avenida)\s+\w+(?:\s+\w+)*)"
neighborhoods = [r"(centro Habana|vedado|boyeros|playa|marianao|bahía|miramar)"]
municipalities = [r"(10 de Octubre|san Miguel|habana del este|santos suárez)"]
nearby = r"(cerca de\s+\w+(?:\s+\w+)*)"
landmarks = r"(Terminal de Ómnibus Nacionales|Plaza de la Revolución|Calixto García|Pediátrico de Centro Habana|Fajardo|Ciudad Deportiva)"

for municipality, areas in json_data.items():
    municipalities.append(re.escape(municipality.lower())) 

    for neighborhood, streets_list in areas.items():
        neighborhoods.append(re.escape(neighborhood.lower())) 
        for street in streets_list:
            streets.append(re.escape(street.lower()))

streets_pattern = "|".join(streets)
neighborhoods_pattern = "|".join(neighborhoods)
municipalities_pattern = "|".join(municipalities)

location_pattern = rf"\b({streets_pattern}|{intersections}|{neighborhoods_pattern}|{municipalities_pattern}|{nearby}|{landmarks})\b"

def find_locations(message):
    matches = re.findall(location_pattern, message, re.IGNORECASE)
    unique_matches = list(set([match[0].strip() if isinstance(match, tuple) else match.strip() for match in matches]))
    return sorted(unique_matches)

data['ubicaciones'] = data['message'].apply(find_locations)

data

,message,precio,moneda,ubicaciones
0,renta x días de apto en el vedado,no especificado,no especificado,[el vedado]
1,no tienes permisos para ejecutar este comando ...,no especificado,no especificado,[]
2,,no especificado,no especificado,[]
8,casa en venta en la zona sur cerca de las fábr...,[2800 ],[usd],"[cerca de las fábricas de cerveza y galleta, h..."
9,busco renta por tiempo indefinido para una par...,[ 20 mil],[mil],[]
...,...,...,...,...
4988,busco alquiler en el vedado límite 150 verde s...,[150 ],no especificado,[el vedado]
4992,"busco alquiler por tiempo indefinido, 58316712",no especificado,no especificado,[]
4993,busco alquiler en la lisa o lo más cerca posible,no especificado,no especificado,[la lisa]
4994,busco alquiler en la lisa,no especificado,no especificado,[la lisa]


Obtención de las coordenadas de los lugares

In [ ]:
CACHE_FILE = "geocache.json"

try:
    with open(CACHE_FILE, "r") as f:
        cache = json.load(f)
except FileNotFoundError:
    cache = {}

def guardar_cache():
    """Guarda el caché actual en un archivo JSON."""
    with open(CACHE_FILE, "w") as f:
        json.dump(cache, f, indent=4)

def obtener_coordenadas(lugar):
    if not lugar:
        return [(None, None)]
    
    geolocator = Nominatim(user_agent="mi_aplicacion_geocodificacion")
    coordenadas = []  # Lista para almacenar las coordenadas obtenidas
    max_retries = 3  # Número máximo de reintentos
    
    for i in lugar:
        # Verificar si el lugar ya está en el caché
        if i in cache:
            print(f"Usando caché para: {i}")
            coordenadas.append(tuple(cache[i]))
            continue
        
        retries = 0
        success = False
        
        while retries < max_retries and not success:
            try:
                location = geolocator.geocode(i + " La Habana, Cuba", timeout=10)
                if location:
                    print(f"Lugar: {i}, Latitud: {location.latitude}, Longitud: {location.longitude}")
                    coordenadas.append((location.latitude, location.longitude))
                    
                    # Guardar en el caché
                    cache[i] = [location.latitude, location.longitude]
                    guardar_cache()
                    
                    success = True
                else:
                    print(f"No se encontró el lugar: {i}.")
                    coordenadas.append((None, None))  
                    success = True  
            except GeocoderTimedOut:
                retries += 1
                print(f"Intento {retries} fallido para '{i}'. Tiempo de espera agotado. Reintentando...")
                time.sleep(2)  
            except GeocoderServiceError as e:
                print(f"Error en el servicio de geocodificación para '{i}': {e}")
                coordenadas.append((None, None))
                success = True  
            except Exception as e:
                print(f"Error inesperado para '{i}': {e}")
                coordenadas.append((None, None))
                success = True  
        
        if not success:  
            print(f"No se pudo obtener la coordenada para '{i}' después de {max_retries} intentos.")
            coordenadas.append((None, None))
        time.sleep(1) 
    
    return coordenadas

In [ ]:
data['locations'] = data['locations'].apply(ast.literal_eval)
coordenadas: List[Tuple[float, float]] = []
for ubicacion in data['locations']:
    
    coords = obtener_coordenadas(ubicacion)
    coordenadas.append(coords)

data['coordenadas'] = coordenadas

data.to_csv(file_path, index=False)

Funcion para buscar ubicaciones cerca de un punto de referencia

In [11]:
def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0

    lat1_rad = math.radians(lat1)
    lon1_rad = math.radians(lon1)
    lat2_rad = math.radians(lat2)
    lon2_rad = math.radians(lon2)

    dlat = lat2_rad - lat1_rad
    dlon = lon2_rad - lon1_rad

    # Fórmula de Haversine
    a = math.sin(dlat / 2)**2 + math.cos(lat1_rad) * math.cos(lat2_rad) * math.sin(dlon / 2)**2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))

    # Distancia en kilómetros
    distance = R * c
    return distance
def location_nearby(radio_inicial_km, incremento_radio_km, max_intentos):
    radio_actual_km = radio_inicial_km
    intentos = 0
    geolocator = Nominatim(user_agent="mi_aplicacion_geocodificacion")
    try:
        referencia = geolocator.geocode(ubicacion_referencia + " La Habana, Cuba", timeout=10)
        if not referencia:
            print(f"No se encontró la ubicación de referencia: {ubicacion_referencia}")
            return []
        lat_ref, lon_ref = referencia.latitude, referencia.longitude
    except Exception as e:
        print(f"Error al obtener las coordenadas de referencia: {e}")
        return []

    while intentos < max_intentos:
        print(f"Buscando ubicaciones dentro de un radio de {radio_actual_km} km...")
        ubicaciones_cercanas = []

        for ubicacion in data['coordenadas']:
            try:
                for i in ubicacion:
                    if i[0] is not None:
                        lat =i[0]
                    if i[-1] is not None:
                        lon=i[-1]
                    distancia = haversine(lat_ref, lon_ref, lat, lon)
                    if distancia <= radio_actual_km:
                        ubicaciones_cercanas.append({
                            "nombre": ubicacion,
                            "distancia_km": round(distancia, 2),
                            "coordenadas": (lat, lon)
                        })
            except Exception as e:
                print(f"Error al procesar la ubicación {ubicacion}: {e}")

        if ubicaciones_cercanas:
            print(f"Se encontraron {len(ubicaciones_cercanas)} ubicaciones cercanas.")
            return ubicaciones_cercanas

        radio_actual_km += incremento_radio_km
        intentos += 1

    print("No se encontraron ubicaciones cercanas después de varios intentos.")
    return []
ubicacion_referencia = "Plaza de la Revolución"

data['coordenadas']=data['coordenadas'].apply(ast.literal_eval)
ubicaciones_cercanas = location_nearby(
    radio_inicial_km=1,  
    incremento_radio_km=1,  
    max_intentos=5  
)
for ubicacion in ubicaciones_cercanas:
    print(f"Ubicación: {ubicacion['nombre']}, Distancia: {ubicacion['distancia_km']} km, Coordenadas: {ubicacion['coordenadas']}")


NameError: name 'data' is not defined

Esta funcion calcula la distancia entre dos puntos y da el tiempo en que te demoras en llegar dependiendo del modo de transporte

In [12]:
API_KEY = "5b3ce3597851110001cf62489c3b3707be9145059892546f17b63840"  
ors_url = "https://api.openrouteservice.org/v2/directions"

def obtener_tiempo_ors(modo, origen, destino, intentos=3, espera=2):

    url = f"{ors_url}/{modo}"
    headers = {"Authorization": API_KEY, "Content-Type": "application/json"}
    payload = {"coordinates": [origen, destino]}

    for intento in range(1, intentos + 1):
        try:
            response = requests.post(url, json=payload, headers=headers, timeout=10)
            
            if response.status_code == 200:
                data = response.json()
                
                if "routes" in data and data["routes"]:
                    duracion_segundos = data["routes"][0]["segments"][0]["duration"]
                    return round(duracion_segundos / 60, 2)  
                
                print(f" Respuesta sin rutas para {modo}: {data}")
                return None
     
            print(f"Intento {intento}/{intentos} - Error {response.status_code}: {response.text}")

        except requests.exceptions.RequestException as e:
            print(f"Intento {intento}/{intentos} - Error al conectarse a la API ({modo}): {e}")
    
        if intento < intentos:
            time.sleep(espera)
    
    print(f" No se pudo obtener la ruta para {modo} después de {intentos} intentos.")
    return None


In [ ]:
# Fórmula de Haversine para calcular la distancia entre dos puntos geográficos
def haversine(lat1, lon1, lat2, lon2):
    # Radio de la Tierra en kilómetros
    R = 6371.0

    # Convertir grados a radianes
    lat1_rad = math.radians(lat1)
    lon1_rad = math.radians(lon1)
    lat2_rad = math.radians(lat2)
    lon2_rad = math.radians(lon2)

    # Diferencias de latitud y longitud
    dlat = lat2_rad - lat1_rad
    dlon = lon2_rad - lon1_rad

    # Fórmula de Haversine
    a = math.sin(dlat / 2)**2 + math.cos(lat1_rad) * math.cos(lat2_rad) * math.sin(dlon / 2)**2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))

    # Distancia en kilómetros
    distance = R * c
    return distance

# Función para encontrar ubicaciones cercanas dentro de un radio 
def encontrar_ubicaciones_cercanas(ubicacion_referencia, lugar, radio_inicial_km, incremento_radio_km=1, max_intentos=5):


    ubi_ref=obtener_coordenadas([ubicacion_referencia])
    for i in ubi_ref:
        if i[0] is not None:
            lat_ref =i[0]
        if i[-1] is not None:
            lon_ref=i[-1]
    ubi=obtener_coordenadas([lugar])
   
    for i in ubi:
        if i[0] is not None:
            lat =i[0]
        if i[-1] is not None:
            lon=i[-1]

    modos = {
    "caminando": "foot-walking",
    "bicicleta": "cycling-regular",
    "carro": "driving-car"
    }
    tiempos = {modo: obtener_tiempo_ors(tipo, [lon_ref, lat_ref], [lon, lat]) for modo, tipo in modos.items()}

    for modo, tiempo in tiempos.items():
        if tiempo is not None:
            print(f"Tiempo en {modo}: {tiempo} minutos desde {ubicacion_referencia} hasta {lugar}.")

    radio_actual_km = radio_inicial_km
    intentos = 0
    while intentos < max_intentos:
        ubicaciones_cercanas = []
        try:
            distancia = haversine(lat_ref, lon_ref, lat, lon)
            if distancia <= radio_actual_km:
                ubicaciones_cercanas.append({
                    "nombre": lugar,
                    "distancia_km": round(distancia, 2),
                    "coordenadas": (lat, lon)
                        })
        except Exception as e:
            print(f"Error al calcular la distancia de {ubicacion}: {e}")

        if ubicaciones_cercanas:
            print(f"Se encontraron {len(ubicaciones_cercanas)} ubicaciones cercanas.")
            return ubicaciones_cercanas

        # Aumentar el radio y continuar buscando
        radio_actual_km += incremento_radio_km
        intentos += 1

    print("No se encontraron ubicaciones cercanas después de varios intentos.")
    return []

# Ubicación de referencia
ubicacion_referencia = 'Universidad de la Habana'
lugar='Centro Habana'


# Encontrar ubicaciones cercanas
ubicaciones_cercanas = encontrar_ubicaciones_cercanas(
    ubicacion_referencia=ubicacion_referencia,
    lugar=lugar,
    radio_inicial_km=2,  # Radio inicial en km
    incremento_radio_km=1,  # Incremento del radio en cada intento
    max_intentos=5  # Máximo número de intentos
)

# Mostrar resultados
for ubicacion in ubicaciones_cercanas:
    print(f"Ubicación: {ubicacion['nombre']}, Distancia: {ubicacion['distancia_km']} km, Coordenadas: {ubicacion['coordenadas']}")

Lugar: Universidad de la Habana, Latitud: 23.13638465, Longitud: -82.3819420021318
Lugar: Centro Habana, Latitud: 23.1415524, Longitud: -82.3598183
Tiempo en caminando: 31.66 minutos desde Universidad de la Habana hasta Centro Habana.
Tiempo en bicicleta: 10.92 minutos desde Universidad de la Habana hasta Centro Habana.
Tiempo en carro: 5.41 minutos desde Universidad de la Habana hasta Centro Habana.
Buscando ubicaciones dentro de un radio de 2 km...
Buscando ubicaciones dentro de un radio de 3 km...
Se encontraron 1 ubicaciones cercanas.
Ubicación: Centro Habana, Distancia: 2.33 km, Coordenadas: (23.1415524, -82.3598183)


#### Ahora vamos a extraer otro feature referente a el tipo de renta (ya sea lineal, por horas, por mes, por dia, por semanas, etc)

In [8]:
def extract_rent_duration(text):
    patterns = {
        "por días": r"(por\s\d+\sdías?|por\s24\s?horas|por\s\d+\s?días?|x\sdías?)",
        "por hora": r"(por\s\d+\shoras?|por\s?hora|por\s?horas)",
        "indefinido": r"(por\stiempo\sindefinido|para\ssiempre)",
        "lineal": r"(al\smes|mensual|por\smes|lineal)",
        "por tiempo limitado": r"(desde\s\d{1,2}(am|pm)?\shasta\s\d{1,2}(am|pm)?|por\ssemanas?|por\stemporadas?)"
    }
    
    for type, pattern in patterns.items():
        if re.search(pattern, text, re.IGNORECASE):
            return type 
    return "no especificado"

data["duration"] = data["message"].apply(extract_rent_duration)
print(data)

                                                message           precio  \
0                     renta x días de apto en el vedado  no especificado   
1     no tienes permisos para ejecutar este comando ...  no especificado   
2                                                        no especificado   
8     casa en venta en la zona sur cerca de las fábr...          [2800 ]   
9     busco renta por tiempo indefinido para una par...        [ 20 mil]   
...                                                 ...              ...   
4988  busco alquiler en el vedado límite 150 verde s...           [150 ]   
4992     busco alquiler por tiempo indefinido, 58316712  no especificado   
4993   busco alquiler en la lisa o lo más cerca posible  no especificado   
4994                          busco alquiler en la lisa  no especificado   
4996  busco alquiler en playa, marianao, lisa hasta ...  no especificado   

               moneda                                        ubicaciones  \
0     no es

In [9]:
def extraer_rent_type(text):
    patterns = {
        "apartamento": r"(apto|apartamento)",
        "casa independiente": r"(casa\sindependiente|casa\b)",
        "habitación": r"(habitación|habitación\sindependiente)",
        "estudio": r"(estudio)"
    }
    
    for type, pattern in patterns.items():
        if re.search(pattern, text, re.IGNORECASE):
            return type 
    
    return "otro" 

data["tipo de renta"] = data["message"].apply(extraer_rent_type)
print(data) 


                                                message           precio  \
0                     renta x días de apto en el vedado  no especificado   
1     no tienes permisos para ejecutar este comando ...  no especificado   
2                                                        no especificado   
8     casa en venta en la zona sur cerca de las fábr...          [2800 ]   
9     busco renta por tiempo indefinido para una par...        [ 20 mil]   
...                                                 ...              ...   
4988  busco alquiler en el vedado límite 150 verde s...           [150 ]   
4992     busco alquiler por tiempo indefinido, 58316712  no especificado   
4993   busco alquiler en la lisa o lo más cerca posible  no especificado   
4994                          busco alquiler en la lisa  no especificado   
4996  busco alquiler en playa, marianao, lisa hasta ...  no especificado   

               moneda                                        ubicaciones  \
0     no es

In [10]:
def extract_number_of_bedrooms(text):
    pattern = r"(\d+/\d+|\d+|un|uno|dos|tres|cuatro|cinco)\s?(cuartos?|habitaciones?|/4?)"
    match = re.search(pattern, text, re.IGNORECASE)
    if match:
        number = match.group(1)
        words_to_numbers = {
            "un": 1, "uno": 1, "dos": 2, "tres": 3,
            "cuatro": 4, "cinco": 5
        }
        if number.lower() in words_to_numbers:
            return words_to_numbers[number.lower()]
        if re.match(r"\d+/\d+", number):
            return int(number.split('/')[0])
        return int(number)
    
    return "no_especificado"

data["bedrooms"] = data["message"].apply(extract_number_of_bedrooms)
print(data)

                                                message           precio  \
0                     renta x días de apto en el vedado  no especificado   
1     no tienes permisos para ejecutar este comando ...  no especificado   
2                                                        no especificado   
8     casa en venta en la zona sur cerca de las fábr...          [2800 ]   
9     busco renta por tiempo indefinido para una par...        [ 20 mil]   
...                                                 ...              ...   
4988  busco alquiler en el vedado límite 150 verde s...           [150 ]   
4992     busco alquiler por tiempo indefinido, 58316712  no especificado   
4993   busco alquiler en la lisa o lo más cerca posible  no especificado   
4994                          busco alquiler en la lisa  no especificado   
4996  busco alquiler en playa, marianao, lisa hasta ...  no especificado   

               moneda                                        ubicaciones  \
0     no es

#### Ya se tienen parte de los features más relevantes. Pero por ahora vamos a enfocarnos un poco en la intención de los posts

In [11]:
def extract_intention(text):
    patterns = {
        "buscar": r"\b(busco|se busca|necesito|se necesita)\b.*?(alquiler|renta|apartamento|casa|cuarto)",
        "rentar": r"\b(rento|renta|alquilo|renta de|se renta|se alquila|disponible|habitación|renta x días)\b.*?(apartamento|casa|cuarto|habitaciones|x días|por días|temporal)",
        "vender": r"\b(vendo|se vende|venta de|casa en venta)\b.*?(apartamento|casa|propiedad|inmueble|cuarto)"
    } 
    for intention, pattern in patterns.items():
        if re.search(pattern, text, re.IGNORECASE):
            return intention
    return "no especificado" 

data["intención"] = data["message"].apply(extract_intention)
print(data)

                                                message           precio  \
0                     renta x días de apto en el vedado  no especificado   
1     no tienes permisos para ejecutar este comando ...  no especificado   
2                                                        no especificado   
8     casa en venta en la zona sur cerca de las fábr...          [2800 ]   
9     busco renta por tiempo indefinido para una par...        [ 20 mil]   
...                                                 ...              ...   
4988  busco alquiler en el vedado límite 150 verde s...           [150 ]   
4992     busco alquiler por tiempo indefinido, 58316712  no especificado   
4993   busco alquiler en la lisa o lo más cerca posible  no especificado   
4994                          busco alquiler en la lisa  no especificado   
4996  busco alquiler en playa, marianao, lisa hasta ...  no especificado   

               moneda                                        ubicaciones  \
0     no es

#### Ahora vamos a intentar extraer las diferentes amenities, como lavadoras, aires acondicionados, balcones, mascotas, internet, etc.

In [ ]:
def extract_amenities(text):
    
    amenity_patterns = {
        "lavadora": r"\blavadora\b",
        "aire_acondicionado": r"\b(aire acondicionado|ac|split)\b",
        "microondas": r"\bmicroondas|microwave\b",
        "mascotas": r"\bmascotas?\b", 
        "balcon": r"\bbalc[oó]n\b",   
        "patio": r"\bpatio|terraza\b",
        "azotea": r"\bazotea\b",
        "refrigerador": r"\b(refrigerador|nevera)\b",
        "ventilador": r"\bventilador\b",
        "internet": r"\binternet\b",
        "barbacoa": r"\bbarbacoa\b",
    }
    
    negation_pattern = r"\b(no|sin|prohibido|no se permite?|no hay|ni|no tiene)\b"
    amenities = {}
    for amenity, pattern in amenity_patterns.items():
        matches = list(re.finditer(pattern, text))
        if matches:
            amenity_value = None
            for m in matches:
                start = m.start()
                window = text[max(0, start - 20):start]
                if re.search(negation_pattern, window):
                    amenity_value = 0
                    break
                else:
                    amenity_value = 1
            amenities[amenity] = amenity_value
        else:
            amenities[amenity] = None
    
    return amenities

amenities_df = data["message"].apply(extract_amenities).apply(pd.Series)
data = data.join(amenities_df)
print(data)

In [12]:
from transformers import TrainingArguments, Trainer, AutoModelForSequenceClassification, AutoTokenizer
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

tokenizer = AutoTokenizer.from_pretrained("dccuchile/bert-base-spanish-wwm-uncased")
label_encoder = LabelEncoder()
data["intención"] = label_encoder.fit_transform(data["intención"]) 
data_for_model = data[["message", "intención"]]

train_df, test_df = train_test_split(data_for_model, test_size=0.2, random_state=42)
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

train_dataset = train_dataset.rename_columns({"intención": "label"})
test_dataset = test_dataset.rename_columns({"intención": "label"})

def tokenize_function(examples):
    tokens = tokenizer(examples["message"], padding="max_length", truncation=True)
    tokens["label"] = examples["label"] 
    return tokens

train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
test_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

num_labels = len(label_encoder.classes_)
model = AutoModelForSequenceClassification.from_pretrained("dccuchile/bert-base-spanish-wwm-uncased", num_labels=num_labels)

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    processing_class=tokenizer
)
trainer.train()



C:\Users\ma907\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Map: 100%|██████████| 502/502 [00:00<00:00, 4259.18 examples/s]
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at dccuchile/bert-base-spanish-wwm-uncased and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss
1,No log,0.238632
2,0.182200,0.151535
3,0.182200,0.197921


TrainOutput(global_step=753, training_loss=0.1311436035085326, metrics={'train_runtime': 8286.8553, 'train_samples_per_second': 0.726, 'train_steps_per_second': 0.091, 'total_flos': 1583430764617728.0, 'train_loss': 0.1311436035085326, 'epoch': 3.0})

In [13]:
from sklearn.metrics import classification_report
import numpy as np

predictions = trainer.predict(test_dataset)
y_pred = np.argmax(predictions.predictions, axis=1)
y_true = np.array(test_dataset["label"])
report = classification_report(y_true, y_pred, target_names=label_encoder.classes_)
print(report)

                 precision    recall  f1-score   support

         buscar       0.97      1.00      0.99       268
no especificado       0.97      0.93      0.95       193
         rentar       0.84      0.90      0.87        40
         vender       1.00      1.00      1.00         1

       accuracy                           0.96       502
      macro avg       0.95      0.96      0.95       502
   weighted avg       0.96      0.96      0.96       502



C:\Users\ma907\AppData\Local\Temp\ipykernel_22316\3086462617.py:6: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  y_true = np.array(test_dataset["label"])


In [14]:
output_dir = "./fine_tuned_intention model"
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
print(f"Modelo guardado en: {output_dir}")

Modelo guardado en: ./fine_tuned_intention model
